# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [ ]:
# Colab only: install Java and pyspark (already provided by the docker image)
import sys
if "google.colab" in sys.modules:
    !apt-get -qq update > /dev/null
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    !pip -q install "pyspark>=4.0"
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [2]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [4]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [5]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [6]:
from pyspark.sql import functions as F

df = (df_trips
      .withColumn("trip_id", F.monotonically_increasing_id())
      .withColumn("duration_min", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
      .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
      .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
      .withColumn("dow", F.date_format("tpep_pickup_datetime", "EEEE")))
df.cache()
df.select("trip_id", "tpep_pickup_datetime", "duration_min", "pickup_hour", "dow").show(5)

+-------+--------------------+------------------+-----------+---------+
|trip_id|tpep_pickup_datetime|      duration_min|pickup_hour|      dow|
+-------+--------------------+------------------+-----------+---------+
|      0| 2019-01-01 00:46:40| 6.666666666666667|          0|  Tuesday|
|      1| 2019-01-01 00:59:47|              19.2|          0|  Tuesday|
|      2| 2018-12-21 13:48:30| 4.166666666666667|         13|   Friday|
|      3| 2018-11-28 15:52:25|3.3333333333333335|         15|Wednesday|
|      4| 2018-11-28 15:56:57|               1.6|         15|Wednesday|
+-------+--------------------+------------------+-----------+---------+
only showing top 5 rows


### Unique key + derived columns
I use `monotonically_increasing_id()` to create `trip_id`. It guarantees a **unique** id but not a consecutive one (the id encodes the partition number), which is enough to identify a trip.
I also add columns reused in the next questions: trip duration in minutes, pickup date, pickup hour and day of week. The dataframe is cached because it is reused many times.

In [7]:
df.orderBy(F.desc("passenger_count")).select("trip_id", "passenger_count", "trip_distance", "fare_amount").show(5)

+-------+---------------+-------------+-----------+
|trip_id|passenger_count|trip_distance|fare_amount|
+-------+---------------+-------------+-----------+
|2012098|            9.0|          0.0|        9.0|
|7286683|            9.0|          0.0|        9.3|
|2883995|            9.0|          0.0|        9.0|
|1296287|            9.0|          0.0|        9.0|
|4534707|            9.0|          0.0|       92.0|
+-------+---------------+-------------+-----------+
only showing top 5 rows


### Highest passenger count
The maximum is **9 passengers**. All the trips shown have a **distance of 0.0**, and one of them costs $92 for 0 miles. A standard yellow cab cannot legally carry 9 passengers, so these values look like data-entry errors from the driver rather than real trips.

In [8]:
df.agg(F.avg("passenger_count")).show()
print("Courses avec 0 passager :", df.filter(F.col("passenger_count") == 0).count())

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+

Courses avec 0 passager : 117381


### Average passenger count
The average is **≈ 1.57 passengers per trip**, so most taxi rides are solo or in pairs.
However, **117,381 trips (~1.5%)** have 0 passengers, which is impossible for a real trip. This is probably a field the driver did not fill in. These zeros slightly pull the average down.

In [9]:
cols = ["trip_id", "trip_distance", "duration_min", "tpep_pickup_datetime", "tpep_dropoff_datetime"]
df.orderBy(F.desc("trip_distance")).select(cols).show(3)
df.filter(F.col("trip_distance") > 0).orderBy("trip_distance").select(cols).show(3)
df.orderBy(F.desc("duration_min")).select(cols).show(3)
df.filter(F.col("duration_min") > 0).orderBy("duration_min").select(cols).show(3)

+-------+-------------+-----------------+--------------------+---------------------+
|trip_id|trip_distance|     duration_min|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+-------------+-----------------+--------------------+---------------------+
|6074091|        831.8|9.483333333333333| 2019-01-25 21:56:39|  2019-01-25 22:06:08|
|4286633|        700.7|6.933333333333334| 2019-01-18 16:32:24|  2019-01-18 16:39:20|
|6770985|       214.01|673.0833333333334| 2019-01-28 17:24:11|  2019-01-29 04:37:16|
+-------+-------------+-----------------+--------------------+---------------------+
only showing top 3 rows
+-------+-------------+-------------------+--------------------+---------------------+
|trip_id|trip_distance|       duration_min|tpep_pickup_datetime|tpep_dropoff_datetime|
+-------+-------------+-------------------+--------------------+---------------------+
|  12564|         0.01|0.06666666666666667| 2019-01-01 00:02:54|  2019-01-01 00:02:58|
|  18828|         0.01| 0.6833333

### Shortest / longest trip
**By distance:**
- Longest: **831.8 miles in 9.5 minutes** (≈ 5,300 mph), then 700.7 miles in 6.9 min. Physically impossible, so these are GPS/meter errors. The 3rd one (214 miles in ~11 h) is plausible (e.g. a long out-of-state ride).
- Shortest (> 0): **0.01 mile**, a few seconds long. Probably a cancelled trip or a meter started and stopped immediately.

**By duration:**
- Longest: **43,648 minutes ≈ 30 days** for 1.2 miles. The meter was obviously never stopped, so this is not a real trip.
- Shortest (> 0): **1 second** with 0 miles.

Conclusion: the raw extremes are dominated by errors, so they must be cleaned before drawing conclusions.

In [10]:
df_jan = df.filter(F.col("pickup_date").between("2019-01-01", "2019-01-31"))
daily = df_jan.groupBy("pickup_date", "dow").count()
daily.orderBy(F.desc("count")).show(3)
daily.orderBy("count").show(3)

+-----------+--------+------+
|pickup_date|     dow| count|
+-----------+--------+------+
| 2019-01-25|  Friday|292499|
| 2019-01-11|  Friday|291714|
| 2019-01-31|Thursday|284625|
+-----------+--------+------+
only showing top 3 rows
+-----------+---------+------+
|pickup_date|      dow| count|
+-----------+---------+------+
| 2019-01-01|  Tuesday|189432|
| 2019-01-21|   Monday|192826|
| 2019-01-02|Wednesday|198737|
+-----------+---------+------+
only showing top 3 rows


### Busiest / slowest single day
Trips dated outside January 2019 are excluded first.
- Busiest: **Friday 2019-01-25 (292,499 trips)**, followed by Friday 01-11 and Thursday 01-31.
- Slowest: **Tuesday 2019-01-01 (189,432 trips)**, i.e. New Year's Day, then **Monday 01-21** (Martin Luther King Jr. Day, a US public holiday) and 01-02 (the day after New Year).

The slowest days are all holidays or the day after one, which makes sense: fewer people commute to work.

In [11]:
df_jan.groupBy("pickup_hour").count().orderBy("pickup_hour").show(24)

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|207758|
|          1|149242|
|          2|109413|
|          3| 78084|
|          4| 61423|
|          5| 75532|
|          6|178598|
|          7|304858|
|          8|373735|
|          9|365924|
|         10|361382|
|         11|375438|
|         12|401172|
|         13|404149|
|         14|433115|
|         15|452679|
|         16|420806|
|         17|468407|
|         18|515374|
|         19|475152|
|         20|423128|
|         21|409873|
|         22|369026|
|         23|281812|
+-----------+------+



### Busiest / slowest time of day
- Peak: **18h (515,374 trips)**, with the whole 17h-19h evening rush above 465k.
- Slowest: **4h (61,423 trips)**, about 8× fewer than the peak.
- A first rise appears at 7h-8h (morning commute), then activity keeps climbing through the afternoon.

The evening is busier than the morning, likely because taxis are used for commuting home *and* for evening leisure (restaurants, bars).

In [12]:
daily.groupBy("dow").agg(F.count("*").alias("nb_jours"), F.avg("count").alias("moyenne_courses")) \
     .orderBy(F.desc("moyenne_courses")).show()

+---------+--------+---------------+
|      dow|nb_jours|moyenne_courses|
+---------+--------+---------------+
|   Friday|       4|       271787.5|
| Thursday|       5|       271398.4|
|Wednesday|       5|       253045.8|
| Saturday|       4|      252494.75|
|  Tuesday|       5|       241815.2|
|   Monday|       4|       226941.0|
|   Sunday|       4|       214972.5|
+---------+--------+---------------+



### Busiest / slowest day of the week (on average)
January 2019 has **5 Tuesdays, Wednesdays and Thursdays but only 4 of the other days**, so a raw count per weekday would be biased. I therefore compute the **average number of trips per date**.
- Busiest: **Friday (≈ 271,800 trips/day)**, very close to Thursday (≈ 271,400).
- Slowest: **Sunday (≈ 215,000 trips/day)**, about 21% below Friday.

Caveat: the Tuesday and Monday averages are pulled down by the holidays (Jan 1 and MLK Day on Jan 21). With only 4-5 days per weekday, one holiday has a large effect.

In [20]:
card = df_jan.filter((F.col("payment_type") == 1) & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) & (F.col("fare_amount") > 0) & (F.col("fare_amount") < 500))
card.select(F.corr("trip_distance", "tip_amount"), F.corr("passenger_count", "tip_amount")).show()
card.groupBy("passenger_count").agg(F.count("*").alias("nb_trips"), F.avg("tip_amount").alias("avg_tip")).orderBy("passenger_count").show()

+-------------------------------+---------------------------------+
|corr(trip_distance, tip_amount)|corr(passenger_count, tip_amount)|
+-------------------------------+---------------------------------+
|             0.7113671620165326|             0.010946793586386939|
+-------------------------------+---------------------------------+

+---------------+--------+------------------+
|passenger_count|nb_trips|           avg_tip|
+---------------+--------+------------------+
|            0.0|   82505| 2.486104720926006|
|            1.0| 3918930|2.5158644910754795|
|            2.0|  777889| 2.591579466994541|
|            3.0|  217731|2.5685921159596994|
|            4.0|   91622|2.5735538407805225|
|            5.0|  230728|  2.60644603169094|
|            6.0|  142618|2.5977789619824776|
|            7.0|       4|13.932500000000001|
|            8.0|      10|            10.956|
|            9.0|       1|               0.0|
+---------------+--------+------------------+



### Does distance or number of passengers affect the tip?
I only keep **credit-card payments** (`payment_type = 1`) because cash tips are not recorded in this dataset (they would appear as 0). I also remove obvious outliers (distance ≥ 100 miles, fare ≤ \$0 or ≥ \$500).
- **Distance:** correlation = **0.71**, a strong positive relationship. Longer trips have a higher fare, and tips are usually a percentage of the fare. The result is almost the same without the outlier filter (0.70), so it is robust.
- **Passengers:** correlation = **0.01**, i.e. no relationship. The average tip stays between \$2.49 and \$2.61 from 0 to 6 passengers, and these groups contain tens of thousands to millions of trips.
- The averages for 7, 8 and 9 passengers (\$13.9, \$11.0, \$0) are based on only **4, 10 and 1 trips**, so they are not meaningful.

Limit: correlation is not causation. Distance mostly affects the tip *through the fare amount*.

In [14]:
df.orderBy(F.desc("extra")).select("trip_id", "extra", "fare_amount", "total_amount", "tpep_pickup_datetime").show(3)

+-------+------+-----------+------------+--------------------+
|trip_id| extra|fare_amount|total_amount|tpep_pickup_datetime|
+-------+------+-----------+------------+--------------------+
|5323483|535.38|  355676.98|   356214.78| 2019-01-23 08:58:09|
|7453230| 23.04|        4.5|       28.34| 2019-01-31 10:06:09|
| 543203|  18.5|       61.0|        92.3| 2019-01-03 18:32:36|
+-------+------+-----------+------------+--------------------+
only showing top 3 rows


### Highest "extra" charge
The highest extra is **\$535.38 (trip 5323483)**, but that trip has a fare of **\$355,676.98**, which is clearly an erroneous record. The next values (\$23.04, \$18.5) are also unusual: `extra` normally only contains small fixed surcharges (e.g. rush-hour and overnight surcharges of \$0.50-\$1).
So the "highest extra" is in reality a data error, not a real charge.

In [33]:
print("Dates hors janvier 2019 :", df.count() - df_jan.count())
print("Durée <= 0 :", df.filter(F.col("duration_min") <= 0).count())
print("Distance = 0 :", df.filter(F.col("trip_distance") == 0).count())
print("Montant négatif :", df.filter(F.col("fare_amount") < 0).count())
df.select("trip_distance", "duration_min", "fare_amount", "tip_amount").describe().show()
df_clean = df_jan.filter((F.col("duration_min") > 0) & (F.col("duration_min") < 1440) & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100) & (F.col("fare_amount") >= 0) & (F.col("fare_amount") < 1000))
print(df_clean.count())

Dates hors janvier 2019 : 537
Durée <= 0 : 6557
Distance = 0 : 55089
Montant négatif : 7129
+-------+------------------+------------------+-----------------+------------------+
|summary|     trip_distance|      duration_min|      fare_amount|        tip_amount|
+-------+------------------+------------------+-----------------+------------------+
|  count|           7696617|           7696617|          7696617|           7696617|
|   mean|2.8301461681153532|16.551081570422276|12.52967677747685|1.8208300763883147|
| stddev| 3.774548394256295| 81.67539611217202|261.5897471783846|2.4994631914320986|
|    min|               0.0|          -84280.5|           -362.0|             -63.5|
|    max|             831.8| 43648.01666666667|        623259.86|            787.25|
+-------+------------------+------------------+-----------------+------------------+

7634874


### Outliers / strange data points
Out of **7,696,617 trips**, I found several types of anomalies:

- **Dates outside January 2019: 537 trips.** Records from other months (even 2018) inside the January file.
- **Duration ≤ 0: 6,557 trips.** A dropoff cannot happen before the pickup (minimum = −84,280 min ≈ −58 days).
- **Distance = 0: 55,089 trips (~0.7%).** Cancelled trips or GPS failures, sometimes still charged (e.g. \$52 for 0 miles).
- **Negative fare: 7,129 trips.** Probably refunds or corrections rather than real trips (minimum = −\$362).
- **0 passengers: 117,381 trips (~1.5%)**, plus a few trips with 9 passengers. Missing input or impossible capacity.
- **Extreme values:** max fare **\$623,259**, max distance **831.8 miles in 9 minutes**, max duration **~30 days**, negative tips (−\$63.5).

The `describe()` output confirms it: the fare's standard deviation (\$261) is about 20× its mean (\$12.5), which shows that a few extreme values distort the statistics.

**Decision:** I build a cleaned dataframe `df_clean` that keeps only January 2019 dates, duration > 0 and < 24 h, distance > 0 and < 100 miles, and fare between \$0 and \$1,000. It keeps **7,634,874 trips (99.2%)**, so only ~0.8% of the data is removed, while averages such as the average fare are no longer biased by absurd records. Trips with 0 passengers are kept because they only affect the passenger-count average.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [34]:
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
response = requests.get(zone_url)
zone_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

zones = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_file)
zones.printSchema()
zones.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


### Loading the taxi zone lookup
Unlike parquet, a CSV file does not contain its schema, so I use `header` and `inferSchema`. The file is small (265 zones), so inferring the schema on the whole file is cheap. Each `LocationID` maps to a zone and a borough.

In [35]:
pu = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("pu_borough"))
do = zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("do_borough"))

trips_z = df_clean.join(F.broadcast(pu), "PULocationID", "left").join(F.broadcast(do), "DOLocationID", "left")
trips_z.cache()
trips_z.select("trip_id", "PULocationID", "pu_borough", "DOLocationID", "do_borough").show(5)

+-------+------------+----------+------------+----------+
|trip_id|PULocationID|pu_borough|DOLocationID|do_borough|
+-------+------------+----------+------------+----------+
|      0|         151| Manhattan|         239| Manhattan|
|      1|         239| Manhattan|         246| Manhattan|
|      7|         163| Manhattan|         229| Manhattan|
|      8|         229| Manhattan|           7|    Queens|
|      9|         141| Manhattan|         234| Manhattan|
+-------+------------+----------+------------+----------+
only showing top 5 rows


### Joining trips with zones
A trip has two locations, so I join the zone table **twice**: once on `PULocationID` (pickup borough) and once on `DOLocationID` (dropoff borough). I use a `left` join to keep all trips, and `broadcast` because the zone table is tiny: Spark sends it to every executor instead of shuffling 7.6 M rows.

In [36]:
trips_z.groupBy("pu_borough").count().orderBy(F.desc("count")).show()
trips_z.groupBy("do_borough").count().orderBy(F.desc("count")).show()

+-------------+-------+
|   pu_borough|  count|
+-------------+-------+
|    Manhattan|6916458|
|       Queens| 457626|
|      Unknown| 151444|
|     Brooklyn|  89466|
|        Bronx|  17210|
|          N/A|   2194|
|Staten Island|    324|
|          EWR|    152|
+-------------+-------+

+-------------+-------+
|   do_borough|  count|
+-------------+-------+
|    Manhattan|6784167|
|       Queens| 326954|
|     Brooklyn| 298198|
|      Unknown| 140954|
|        Bronx|  57107|
|          N/A|  14822|
|          EWR|  10525|
|Staten Island|   2147|
+-------------+-------+



### Borough with the most pickups / dropoffs
- **Manhattan dominates both**: 6.92 M pickups (**~91%**) and 6.78 M dropoffs (~89%). Queens is far behind (458 k pickups).
- There is a clear asymmetry for the other boroughs: **Brooklyn has 298 k dropoffs but only 89 k pickups**, and **EWR (Newark airport) has 10.5 k dropoffs but only 152 pickups**. Yellow taxis bring people *out* of Manhattan but rarely pick up outside it. A plausible explanation (not verified here) is that outer boroughs are served by other services (green cabs, ride-hailing apps).
- "Unknown" and "N/A" are location codes with no real borough (~150 k pickups), which is another data-quality limit.

In [37]:
from pyspark.sql.window import Window

by_hour = trips_z.groupBy("pu_borough", "pickup_hour").count()
w_busy = Window.partitionBy("pu_borough").orderBy(F.desc("count"))
w_slow = Window.partitionBy("pu_borough").orderBy("count")

(by_hour.withColumn("rank_busy", F.row_number().over(w_busy))
        .withColumn("rank_slow", F.row_number().over(w_slow))
        .filter((F.col("rank_busy") == 1) | (F.col("rank_slow") == 1))
        .withColumn("type", F.when(F.col("rank_busy") == 1, "busiest").otherwise("slowest"))
        .select("pu_borough", "type", "pickup_hour", "count")
        .orderBy("pu_borough", "type").show(20))

+-------------+-------+-----------+------+
|   pu_borough|   type|pickup_hour| count|
+-------------+-------+-----------+------+
|        Bronx|busiest|          7|  1727|
|        Bronx|slowest|          3|   200|
|     Brooklyn|busiest|          8|  6805|
|     Brooklyn|slowest|          3|  1847|
|          EWR|busiest|         15|    23|
|          EWR|slowest|          4|     1|
|    Manhattan|busiest|         18|469543|
|    Manhattan|slowest|          4| 52854|
|          N/A|busiest|         14|   136|
|          N/A|slowest|          4|    46|
|       Queens|busiest|         21| 28943|
|       Queens|slowest|          3|  2919|
|Staten Island|busiest|          8|    34|
|Staten Island|slowest|         22|     2|
|      Unknown|busiest|         18| 10310|
|      Unknown|slowest|          4|  1214|
+-------------+-------+-----------+------+



### Busy / slow times by borough (pickup borough)
- **Manhattan**: busiest at **18h** (evening rush), slowest at 4h, the same pattern as the global data since Manhattan is 91% of it.
- **Brooklyn (8h) and Bronx (7h)** peak in the **morning**, consistent with residents commuting to work.
- **Queens** peaks at **21h**. Queens contains the JFK and LaGuardia airports, so evening flight arrivals are a likely explanation (hypothesis, not tested).
- Almost every borough is slowest around **3h-4h**.
- EWR and Staten Island have very few trips per hour (1 to 34), and N/A is not a real borough, so their peaks are not meaningful.

In [38]:
daily_b = trips_z.groupBy("pu_borough", "pickup_date", "dow").count()
avg_dow_b = daily_b.groupBy("pu_borough", "dow").agg(F.round(F.avg("count"), 1).alias("avg_trips"))
w = Window.partitionBy("pu_borough").orderBy(F.desc("avg_trips"))
avg_dow_b.withColumn("rk", F.row_number().over(w)).filter("rk = 1").drop("rk").orderBy(F.desc("avg_trips")).show()

+-------------+---------+---------+
|   pu_borough|      dow|avg_trips|
+-------------+---------+---------+
|    Manhattan|   Friday| 245023.0|
|       Queens|   Monday|  16186.8|
|      Unknown| Thursday|   5505.8|
|     Brooklyn|   Friday|   3191.3|
|        Bronx|   Friday|    638.8|
|          N/A|Wednesday|     77.4|
|Staten Island|   Friday|     15.3|
|          EWR|   Friday|      8.8|
+-------------+---------+---------+



### Busiest day of the week by borough (average per date)
As in Part 1, I use the average number of trips per date to avoid the 4-vs-5 weekday bias.
- **Friday** is the busiest day for Manhattan, Brooklyn, Bronx, Staten Island and EWR.
- **Queens is the exception, with Monday**, even though Monday includes MLK Day. A possible explanation is people returning from weekend trips via the airports (hypothesis).
- The results for EWR and Staten Island rely on fewer than 20 trips per day, and N/A / Unknown are not real boroughs, so they are not reliable.

In [39]:
trips_z.groupBy("pu_borough").agg(
    F.count("*").alias("nb_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare")
).orderBy(F.desc("nb_trips")).show()

+-------------+--------+------------+--------+
|   pu_borough|nb_trips|avg_distance|avg_fare|
+-------------+--------+------------+--------+
|    Manhattan| 6916458|        2.24|   10.63|
|       Queens|  457626|       11.61|   35.69|
|      Unknown|  151444|        2.55|   11.71|
|     Brooklyn|   89466|        4.91|   18.83|
|        Bronx|   17210|        7.57|    26.9|
|          N/A|    2194|         5.4|   38.43|
|Staten Island|     324|       13.92|   44.54|
|          EWR|     152|        7.75|   71.05|
+-------------+--------+------------+--------+



### Average trip distance and fare by borough (pickup borough)
- **Manhattan** has the shortest and cheapest trips: **2.24 miles, \$10.63** on average. Taxis are mostly used for short intra-Manhattan rides.
- **Queens**: **11.61 miles, \$35.69**, most likely because of airport trips to/from JFK and LaGuardia.
- **Staten Island** (13.92 miles, \$44.54) and **EWR** (\$71.05) have the longest/most expensive trips, but they are based on very few trips (324 and 152).
- Distance and fare move together, which is consistent with a metered fare based on distance and time.

In [40]:
raw_z = df.join(F.broadcast(pu), "PULocationID", "left").join(F.broadcast(do), "DOLocationID", "left")
sel = ["trip_id", "fare_amount", "trip_distance", "duration_min", "pu_borough", "do_borough"]
raw_z.orderBy(F.desc("fare_amount")).select(sel).show(3)
raw_z.orderBy("fare_amount").select(sel).show(3)
trips_z.orderBy(F.desc("fare_amount")).select(sel).show(3)
trips_z.filter("fare_amount > 0").orderBy("fare_amount").select(sel).show(3)

+-------+-----------+-------------+------------+----------+----------+
|trip_id|fare_amount|trip_distance|duration_min|pu_borough|do_borough|
+-------+-----------+-------------+------------+----------+----------+
|2499655|  623259.86|          2.4|        19.9| Manhattan| Manhattan|
|5323483|  355676.98|          0.0|         0.0| Manhattan|   Unknown|
|2159971|    36090.3|          0.0|         0.0|   Unknown|   Unknown|
+-------+-----------+-------------+------------+----------+----------+
only showing top 3 rows
+-------+-----------+-------------+-------------------+----------+----------+
|trip_id|fare_amount|trip_distance|       duration_min|pu_borough|do_borough|
+-------+-----------+-------------+-------------------+----------+----------+
|4890649|     -362.0|          0.0|               6.85|    Queens|    Queens|
|6308200|     -320.0|          0.0|0.13333333333333333|       N/A|       N/A|
|  57094|     -300.0|          0.0|  4.183333333333334|     Bronx|     Bronx|
+-------+--

### Highest / lowest fares and associated borough
**Raw data:**
- Highest: **\$623,259.86** for 2.4 miles (**Manhattan → Manhattan**), then \$355,677 (Manhattan → Unknown). These are obviously data-entry errors.
- Lowest: **−\$362** (**Queens → Queens**), then −\$320 (N/A) and −\$300 (Bronx), all with 0 miles. Probably refunds or corrections.

**Cleaned data (fare capped at \$1,000):**
- Highest: **\$684 (Manhattan → Manhattan)**, but for 0.1 mile in 46 seconds, so still suspicious. The next ones (\$679.50 in the Bronx, \$519 Manhattan → N/A) last 17 to 22 hours, which suggests the meter was left running.
- Lowest positive fare: **\$0.01** for 15.1 miles (Manhattan → N/A), which is not realistic either.

Conclusion: even after cleaning, the extreme fares are mostly errors. A perfect cleaning would need more rules (e.g. price per mile), but these few records have no visible impact on the averages of millions of trips.

In [41]:
for year in [2026, 2025]:
    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-01.parquet'
    response = requests.get(url)
    print(year, response.status_code)
    if response.status_code == 200:
        latest_file = f"yellow_tripdata_{year}-01.parquet"
        with open(latest_file, "wb") as f:
            f.write(response.content)
        latest_year = year
        break

df_new = spark.read.parquet(latest_file)
df_new.printSchema()

2026 200
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



### Loading the most recent January
The code tries January 2026 first, then 2025. **January 2026 is available** (HTTP 200), so it is used for the comparison.
The schema has changed since 2019: timestamps are now `timestamp_ntz`, `passenger_count` is a `long`, and there are new columns (`Airport_fee`, `cbd_congestion_fee`). That is why I do not merge the two datasets: I compute the same metrics separately on each, with the **same cleaning rules**.

In [42]:
def clean(d, year):
    return (d.withColumn("duration_min", (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
             .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
             .filter(F.col("pickup_date").between(f"{year}-01-01", f"{year}-01-31"))
             .filter((F.col("duration_min") > 0) & (F.col("duration_min") < 1440)
                     & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100)
                     & (F.col("fare_amount") >= 0) & (F.col("fare_amount") < 1000)))

def metrics(d, label):
    return d.agg(F.lit(label).alias("dataset"),
                 F.count("*").alias("nb_trips"),
                 F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
                 F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
                 F.round(F.avg("duration_min"), 1).alias("avg_duration_min"),
                 F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                 F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
                 F.round(F.avg("total_amount"), 2).alias("avg_total"))

new_clean = clean(df_new, latest_year)
metrics(df_clean, "2019-01").unionByName(metrics(new_clean, f"{latest_year}-01")).show()

+-------+--------+--------------+------------+----------------+--------+-------+---------+
|dataset|nb_trips|avg_passengers|avg_distance|avg_duration_min|avg_fare|avg_tip|avg_total|
+-------+--------+--------------+------------+----------------+--------+-------+---------+
|2019-01| 7634874|          1.57|        2.85|            16.6|    12.3|   1.81|    15.57|
|2026-01| 3518350|          1.25|        3.49|            17.5|   21.07|   2.67|    29.66|
+-------+--------+--------------+------------+----------------+--------+-------+---------+



### January 2019 vs January 2026 (same cleaning on both)
- **Trips:** 7.63 M → 3.52 M (**−54%**)
- **Avg passengers:** 1.57 → 1.25 (−20%)
- **Avg distance:** 2.85 → 3.49 miles (+22%)
- **Avg duration:** 16.6 → 17.5 min (+5%)
- **Avg fare:** \$12.30 → \$21.07 (**+71%**)
- **Avg tip:** \$1.81 → \$2.67 (+48%)
- **Avg total:** \$15.57 → \$29.66 (**+90%**)

Interpretation:
- **Yellow taxi usage has more than halved.** The data alone cannot explain why. Plausible factors are competition from ride-hailing apps and post-Covid changes in commuting.
- **Prices rose much faster than distance** (+71% fare vs +22% distance), so the price per mile increased, which suggests fare increases and inflation.
- **The total grows faster than the fare**: the gap between total and fare goes from \$3.27 to \$8.59. This reflects surcharges added since 2019, such as the new `cbd_congestion_fee` column visible in the 2026 schema.

In [43]:
new_z = new_clean.join(F.broadcast(pu), "PULocationID", "left")
old_b = trips_z.groupBy("pu_borough").agg(F.count("*").alias("trips_2019"), F.round(F.avg("fare_amount"), 2).alias("fare_2019"))
new_b = new_z.groupBy("pu_borough").agg(F.count("*").alias(f"trips_{latest_year}"), F.round(F.avg("fare_amount"), 2).alias(f"fare_{latest_year}"))
old_b.join(new_b, "pu_borough", "outer").orderBy(F.desc("trips_2019")).show()

+-------------+----------+---------+----------+---------+
|   pu_borough|trips_2019|fare_2019|trips_2026|fare_2026|
+-------------+----------+---------+----------+---------+
|    Manhattan|   6916458|    10.63|   3010456|    17.53|
|       Queens|    457626|    35.69|    320757|    48.48|
|      Unknown|    151444|    11.71|      4086|    20.37|
|     Brooklyn|     89466|    18.83|    146492|    30.85|
|        Bronx|     17210|     26.9|     35300|    32.65|
|          N/A|      2194|    38.43|       688|    73.64|
|Staten Island|       324|    44.54|       455|     41.7|
|          EWR|       152|    71.05|       116|    95.09|
+-------------+----------+---------+----------+---------+



### Comparison by borough
- **Manhattan collapses**: 6.92 M → 3.01 M trips (**−56%**), and Queens drops by 30%.
- **Brooklyn (+64%) and the Bronx (×2) grow**, so yellow taxi activity is slightly less concentrated in Manhattan than in 2019 (91% → 86% of pickups).
- Average fares rise by **+65% in Manhattan**, +64% in Brooklyn and +36% in Queens.
- "Unknown" pickups drop from 151 k to 4 k, which suggests the location data quality improved.
- Staten Island, EWR and N/A remain too small (< 1,000 trips) to draw conclusions.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [44]:
df_jan.createOrReplaceTempView("trips_jan")
df_clean.createOrReplaceTempView("trips_clean")
zones.createOrReplaceTempView("zones")

### Registering the dataframes as SQL views
To query the dataframes with pure SQL, I register them as temporary views: `trips_jan` (January 2019 trips, raw), `trips_clean` (cleaned trips) and `zones` (zone lookup). The three questions below were answered with the DataFrame API in Parts 1 and 2. I redo them in SQL to check that both approaches give the same result.

In [45]:
spark.sql("""
    WITH daily AS (
        SELECT pickup_date, dow, COUNT(*) AS nb
        FROM trips_jan
        GROUP BY pickup_date, dow
    )
    SELECT dow, COUNT(*) AS nb_days, ROUND(AVG(nb), 1) AS avg_trips
    FROM daily
    GROUP BY dow
    ORDER BY avg_trips DESC
""").show()

+---------+-------+---------+
|      dow|nb_days|avg_trips|
+---------+-------+---------+
|   Friday|      4| 271787.5|
| Thursday|      5| 271398.4|
|Wednesday|      5| 253045.8|
| Saturday|      4| 252494.8|
|  Tuesday|      5| 241815.2|
|   Monday|      4| 226941.0|
|   Sunday|      4| 214972.5|
+---------+-------+---------+



### SQL 1: busiest / slowest day of the week (on average)
The CTE `daily` counts trips per date. The outer query then averages these counts per weekday, which avoids the 4-vs-5 weekday bias.
The result is **identical to Part 1**: Friday is the busiest (271,787.5 trips/day) and Sunday the slowest (214,972.5). The only difference is Saturday (252,494.8 vs 252,494.75), which comes from `ROUND(…, 1)` in the SQL query.

In [46]:
spark.sql("""
    SELECT trip_id, extra, fare_amount, total_amount, tpep_pickup_datetime
    FROM trips_jan
    WHERE extra = (SELECT MAX(extra) FROM trips_jan)
""").show()

+-------+------+-----------+------------+--------------------+
|trip_id| extra|fare_amount|total_amount|tpep_pickup_datetime|
+-------+------+-----------+------------+--------------------+
|5323483|535.38|  355676.98|   356214.78| 2019-01-23 08:58:09|
+-------+------+-----------+------------+--------------------+



### SQL 2: highest "extra" charge
A subquery computes `MAX(extra)` and the outer query returns the matching trip. The result is **identical to Part 1**: trip **5323483** with an extra of **\$535.38**, on an erroneous fare of \$355,676.98.
Unlike `orderBy(...).show(3)` in PySpark, this query returns *every* trip tied at the maximum. Here there is only one.

In [47]:
spark.sql("""
    SELECT z.Borough AS pu_borough,
           COUNT(*) AS nb_trips,
           ROUND(AVG(t.trip_distance), 2) AS avg_distance,
           ROUND(AVG(t.fare_amount), 2) AS avg_fare
    FROM trips_clean t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY nb_trips DESC
""").show()

+-------------+--------+------------+--------+
|   pu_borough|nb_trips|avg_distance|avg_fare|
+-------------+--------+------------+--------+
|    Manhattan| 6916458|        2.24|   10.63|
|       Queens|  457626|       11.61|   35.69|
|      Unknown|  151444|        2.55|   11.71|
|     Brooklyn|   89466|        4.91|   18.83|
|        Bronx|   17210|        7.57|    26.9|
|          N/A|    2194|         5.4|   38.43|
|Staten Island|     324|       13.92|   44.54|
|          EWR|     152|        7.75|   71.05|
+-------------+--------+------------+--------+



### SQL 3 (with a JOIN): trips, average distance and fare by pickup borough
I join the cleaned trips with the zone lookup (`LEFT JOIN zones ON PULocationID = LocationID`) and aggregate by borough. The result is **identical to Part 2**: Manhattan has 6,916,458 pickups with the shortest (2.24 miles) and cheapest (\$10.63) trips, while Queens has much longer and more expensive trips (11.61 miles, \$35.69).

**Conclusion:** the DataFrame API and Spark SQL give the same results, because both go through the same engine: Spark's Catalyst optimizer turns them into an optimized execution plan. The choice between them is mainly about readability: SQL is more concise for joins and aggregations, while the DataFrame API is easier to chain and reuse in Python (functions, variables, loops).

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing